# Ensemble Class - Basic Usage

**Status**: Example/testing notebook - not yet validated with real data

This notebook demonstrates the modernized Ensemble class salvaged from evac.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # Add parent dir to path

import datetime
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from ..ensemble import Ensemble

# Check if Herbie available
try:
    from herbie import Herbie
    print("✓ Herbie available")
except ImportError:
    print("✗ Herbie not available - install with: pip install herbie-data")

## Example 1: Load GEFS Ensemble

Load a small GEFS ensemble for recent date.

In [ ]:
# Configuration
init_time = datetime.datetime(2025, 10, 26, 0)  # Adjust to recent date
members = range(1, 6)  # First 5 perturbation members
forecast_hours = range(0, 24, 6)  # 0-24h every 6h
variables = ['TMP:2 m']  # 2-meter temperature

print(f"Loading GEFS from {init_time}...")
print(f"Members: {list(members)}")
print(f"Forecast hours: {list(forecast_hours)}")
print(f"Variables: {variables}")

# This will attempt to download data - may take a few minutes
# Uncomment when ready to test with real data:
# ens = Ensemble.from_gefs(
#     init_time=init_time,
#     members=members,
#     forecast_hours=forecast_hours,
#     variables=variables
# )
# print(ens)

## Example 2: Create Mock Ensemble (for testing without download)

Create a synthetic ensemble to test functionality without needing real data.

In [ ]:
# Create mock ensemble for testing
n_members = 10
n_times = 5
n_lats = 20
n_lons = 30

# Generate synthetic data
# Base pattern + random perturbations (simulating ensemble spread)
lats = np.linspace(38, 42, n_lats)
lons = np.linspace(-112, -108, n_lons)
times = [datetime.datetime(2025, 10, 26, h) for h in range(0, n_times * 6, 6)]
members = [f'p{i:02d}' for i in range(1, n_members + 1)]

# Create base temperature field (decreasing northward)
lat_grid, lon_grid = np.meshgrid(lats, lons, indexing='ij')
base_temp = 290 - (lat_grid - 38) * 2  # Cooler to the north

# Add time evolution and member perturbations
data = np.zeros((n_members, n_times, n_lats, n_lons))
for m in range(n_members):
    for t in range(n_times):
        # Add random perturbation per member
        member_perturbation = np.random.randn(n_lats, n_lons) * 2
        # Add diurnal cycle
        diurnal = 5 * np.sin(2 * np.pi * t / n_times)
        data[m, t, :, :] = base_temp + member_perturbation + diurnal

# Create xarray Dataset
ds_mock = xr.Dataset(
    {
        't2m': (['member', 'time', 'lat', 'lon'], data, {
            'long_name': '2-meter temperature',
            'units': 'K',
            'standard_name': 'air_temperature'
        })
    },
    coords={
        'member': members,
        'time': times,
        'lat': lats,
        'lon': lons,
        'init_time': datetime.datetime(2025, 10, 26, 0)
    }
)

# Create Ensemble object
ens_mock = Ensemble(ds_mock, lazy=False)
print("Mock ensemble created:")
print(ens_mock)

## Example 3: Ensemble Statistics

In [ ]:
# Compute ensemble mean
mean = ens_mock.mean('t2m')
print(f"Mean shape: {mean.shape}")
print(f"Mean range: {mean.values.min():.2f} - {mean.values.max():.2f} K")

# Compute ensemble spread
spread = ens_mock.std('t2m')
print(f"\nSpread shape: {spread.shape}")
print(f"Spread range: {spread.values.min():.2f} - {spread.values.max():.2f} K")

# Compute percentiles
p10 = ens_mock.percentile('t2m', 10)
p90 = ens_mock.percentile('t2m', 90)
print(f"\n10th-90th percentile range: {p10.values.min():.2f} - {p90.values.max():.2f} K")

## Example 4: Exceedance Probability (salvaged from evac)

In [ ]:
# Probability of temperatures below freezing (273.15 K)
prob_freeze = ens_mock.get_exceedance_prob(
    var='t2m',
    threshold=273.15,
    operator='<'
)

print(f"Freezing probability shape: {prob_freeze.shape}")
print(f"Probability range: {prob_freeze.values.min():.1f}% - {prob_freeze.values.max():.1f}%")

# Plot at first time step
plt.figure(figsize=(10, 6))
prob_freeze.isel(time=0).plot(cmap='RdYlBu_r', vmin=0, vmax=100)
plt.title(f'Probability of T < 0°C\n{times[0]}')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

## Example 5: Representative Member (salvaged from evac)

In [ ]:
# Find member closest to mean at specific time
repr_member = ens_mock.closest_to_mean(
    var='t2m',
    time=times[2]  # Middle time
)

print(f"Member closest to mean: {repr_member}")

# Compare representative member to mean
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Representative member
ens_mock.ds['t2m'].sel(member=repr_member, time=times[2]).plot(
    ax=axes[0], cmap='RdYlBu_r'
)
axes[0].set_title(f'Representative Member ({repr_member})')

# Ensemble mean
ens_mock.mean('t2m', time=times[2]).plot(
    ax=axes[1], cmap='RdYlBu_r'
)
axes[1].set_title('Ensemble Mean')

plt.tight_layout()
plt.show()

## Example 6: Web API - Time Series JSON

In [ ]:
# Export time series at a point for BasinWx
location = (40.0, -110.0)  # Uinta Basin point

# All members
json_all = ens_mock.to_timeseries_json(
    var='t2m',
    location=location,
    members='all'
)

print("JSON structure (all members):")
print(f"  Location: {json_all['location']}")
print(f"  Variable: {json_all['variable']}")
print(f"  Times: {len(json_all['times'])} time steps")
print(f"  Members: {list(json_all['members'].keys())}")

# Mean and spread (for simpler visualization)
json_spread = ens_mock.to_timeseries_json(
    var='t2m',
    location=location,
    members='spread'
)

print("\nJSON structure (spread):")
print(f"  Members: {list(json_spread['members'].keys())}")

# Visualize
plt.figure(figsize=(10, 6))
mean_vals = np.array(json_spread['members']['mean'])
upper_vals = np.array(json_spread['members']['upper'])
lower_vals = np.array(json_spread['members']['lower'])
time_indices = range(len(mean_vals))

plt.plot(time_indices, mean_vals, 'k-', linewidth=2, label='Mean')
plt.fill_between(time_indices, lower_vals, upper_vals, alpha=0.3, label='Mean ± 1σ')
plt.xlabel('Forecast Hour')
plt.ylabel('Temperature (K)')
plt.title(f'Ensemble Forecast at {location}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Example 7: Iteration (salvaged from evac)

In [ ]:
# Iterate over members
print("Member iteration:")
for member in ens_mock:
    member_data = ens_mock.ds['t2m'].sel(member=member)
    mean_temp = member_data.mean().values
    print(f"  {member}: mean T = {mean_temp:.2f} K")

## Next Steps

1. **Test with real GEFS data**: Uncomment Example 1 and run with recent date
2. **Benchmark performance**: Compare lazy vs eager loading for large ensemble
3. **Validate calculations**: Compare exceedance probabilities with manual calculations
4. **Test lagged ensemble**: Create `lagged_ensemble.ipynb` example
5. **Web integration**: Coordinate JSON format with BasinWx frontend team